# Overlap and divergence of anomaly scores

In [1]:
import os
# Set environment variables to disable multithreading
# as users will probably want to set the number of cores
# to the max of their computer.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from anomaly.constants import GALAXY_LINES
from anomaly.utils import specobjid_to_idx
from anomaly.utils import VelocityFilter
from anomaly.utils import AnomalyOverlapAnalyzer
from autoencoders.ae import AutoEncoder

from sdss.metadata import MetaData

meta = MetaData()

# Constants

# Custom functions

## IDs top anomalies

In [3]:
def get_ids_set(score, df, quantile=99, n_top=None, use_ntop=False):

    if use_ntop is False:
        
        quantile *= 0.01
        thresh = df[score].quantile(quantile)
        ids = set(df[df[score] > thresh].index)
        
    else:

        ids = set(
            df[score].sort_values(
                ascending=False
            ).iloc[:n_top].index
        )

    return ids

## Figures

In [49]:
def anomaly_plot(wave, specs, objids, ranks, save_to, figsize=(16, 6)):

    fig, ax = plt.subplots(
        figsize=figsize
    )

    for spec, objid, rank in zip(specs, objids, ranks):

        print(f'Rank {rank:03d}', end='\r')

        ax.clear()

        ax.plot(wave, spec, color="black", label=f'Rank: {rank}')

        ax.minorticks_on()
        ax.set_xlabel(r"$\lambda$ [nm]")
        ax.set_title(f"Object ID: {objid}")

        ax.legend(
            loc='upper left',
            frameon=False,
        )

        fig.savefig(
            f"{save_to}/{rank:03d}_{objid}.jpeg",
            bbox_inches='tight'
        )

    plt.close(fig)

In [51]:
def common_anomaly_plot(wave, specs, objids, ranks_lof, ranks_iforest, save_to, figsize=(16, 6)):

    fig, ax = plt.subplots(
        figsize=figsize
    )

    for (
        spec, objid, rank_lof, rank_iforest
    ) in zip(specs, objids, ranks_lof, ranks_iforest):

        print(
            f'Rank LOF, iForest: {rank_lof:03d}, {rank_iforest:03d}',
            end='\r'
        )

        ax.clear()

        ax.plot(
            wave, spec, color="black",
            label=f'Rank LOF, iForest: {rank_lof}, {rank_iforest}'
        )

        ax.minorticks_on()
        ax.set_xlabel(r"$\lambda$ [nm]")
        ax.set_title(f"Object ID: {objid}")

        ax.legend(
            loc='upper left',
            frameon=False,
        )

        fig.savefig(
            f"{save_to}/lof_{rank_lof:03d}_iForest_{rank_iforest:03d}_{objid}.jpeg",
            bbox_inches='tight'
        )

    plt.close(fig)

# Config

## Directories

In [35]:
phd_dir = "/home/elom/phd"
thesis_dir = f"{phd_dir}/thesis"
ch4_dir = f"{thesis_dir}/chapters/04_figures"
data_dir = f"{phd_dir}/code"
spectra_dir = f"{data_dir}/spectra"
models_dir = f"{data_dir}/models"
latent_dir = f"{data_dir}/latent"
bins_ids = [f'bin_{i:02d}' for i in range(4)] 

## Data

In [5]:
wave = np.load(f"{spectra_dir}/wave_spectra_imputed.npy")
wave_nm = wave*0.1

spectra = np.load(
    f"{spectra_dir}/spectra_imputed.npy",
    mmap_mode="r"
)

final_meta_df = pd.read_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop.csv.gz",
    index_col="specobjid",
)

idx_id_spec = np.load(
    f"{spectra_dir}/ids_imputing.npy",
    mmap_mode='r'
)

## iforest scores

In [14]:
iforest_scores_df_dict = {}
# iforest_model_dict = {}
iforest_params = {
    'bin_00': 's256_e300_f100',
    'bin_01': 's256_e300_f75',
    'bin_02': 's256_e300_f100',
    'bin_03': 's128_e300_f100'
}

for bin_id in bins_ids:

    # load df of scores
    _df = pd.read_csv(
        f"{latent_dir}/{bin_id}/iforest/"
        f"iforest_scores_{bin_id}_{iforest_params[bin_id]}.csv",
        index_col='specobjid'
    )

    _df.columns = ['raw_iforest', 'iforest_score', 'rank_iforest']
    iforest_scores_df_dict[bin_id] = _df

In [15]:
bin_id = 'bin_03'
iforest_scores_df_dict[bin_id].sort_values(by='rank_iforest', ascending=True).head()

,raw_iforest,iforest_score,rank_iforest
specobjid,,,
969534273977608192,-0.172210,1.000000,1
1538002139001939968,-0.168068,0.988482,2
561847176611784704,-0.166951,0.985377,3
2829471472643237888,-0.157599,0.959367,4
2811439757043722240,-0.148941,0.935291,5


## LOF scores

In [36]:
lof_scores_df_dict = {}
# lof_model_dict = {}
n = 60
metric = 'manhattan'

for bin_id in bins_ids:

    # load df of scores
    _df = pd.read_csv(
        f"{latent_dir}/{bin_id}/lof/"
        f"lof_scores_{bin_id}_n{n}_{metric}.csv",
        index_col='specobjid'
    )

    _df.columns = ['lof', 'lof_score', 'rank_lof']
    lof_scores_df_dict[bin_id] = _df
    
    # # load models
    # load_from = os.path.join(
    #     latent_dir, bin_id,
    #     f'lof_{bin_id}_n{n}_{metric}.joblib'
    # )

    # model = joblib.load(load_from)
    # lof_model_dict[bin_id] = model

In [37]:
bin_id = 'bin_01'
lof_scores_df_dict[bin_id].sort_values(by='rank_lof', ascending=True).head()

,lof,lof_score,rank_lof
specobjid,,,
2034543258029287424,-4.606827,1.000000,1
650786678071388160,-4.522794,0.981759,2
1857818737302857728,-3.762106,0.816637,3
1099043276306016256,-3.408893,0.739965,4
1623712368703858688,-3.286496,0.713397,5


## Join scores

In [38]:
scores_df_dict = {}

for bin_id in bins_ids:

    _df = lof_scores_df_dict[bin_id].join(
        iforest_scores_df_dict[bin_id],
        how='inner'
    ).copy()

    scores_df_dict[bin_id] = _df


In [39]:
bin_id = 'bin_00'
scores_df_dict[bin_id].sort_values(by='rank_lof', ascending=True).head()

,lof,lof_score,rank_lof,raw_iforest,iforest_score,rank_iforest
specobjid,,,,,,
407585970297268224,-8.440069,1.000000,1,-0.119439,0.813470,342
2367797523948529664,-5.663374,0.671010,2,-0.180470,0.949756,39
2934148553238407168,-4.983568,0.590465,3,-0.177416,0.942936,49
2278872322632869888,-4.601294,0.545173,4,-0.179087,0.946667,42
1079775709385222144,-4.409762,0.522479,5,-0.182996,0.955396,31


# IDs per score
Get specobjid for top 1\% of anomalies of each score

In [29]:
bin_id = 'bin_03'
score_df = scores_df_dict[bin_id]

ids_top_dict = {}

all_scores = ['lof_score', 'iforest_score']

for score in all_scores:

    ids_top_dict[score] = get_ids_set(
        score=score,
        df=score_df.copy(),
        quantile=99,
        n_top=1000,
        use_ntop=False
    )

In [30]:
[
    n for n 
    in [len(v) for v in ids_top_dict.values()]
]

[1819, 1819]

# Pair wise overlap

In [44]:
for bin_id in bins_ids:

    score_df = scores_df_dict[bin_id]

    (
        ids_lof, ids_iforest,
        common_ids,
        only_in_lof_ids, only_in_iforest_ids
    ) = AnomalyOverlapAnalyzer.overlap_pair_scores(
        score_a='lof_score',
        score_b='iforest_score',
        df=score_df.copy(),
        quantile=99
    )

    print(bin_id) 
    ids_a_b = set(list(ids_lof) + list(ids_iforest))
    n_ids_a_b = len(ids_a_b)
    print(f"N in A or B: {n_ids_a_b}")
    # 
    n_common = len(common_ids)
    print(f"N common: {n_common}")
    #
    overlap_pct = (n_common/n_ids_a_b)*100
    print(f"Overlap: {overlap_pct:.2f}%")
    n_ids_only_a = len(only_in_a)
    print(f"N only in A (B): {n_ids_only_a}")
    print('-'*50)

bin_00
N in A or B: 3293
N common: 345
Overlap: 10.48%
N only in A (B): 1301
--------------------------------------------------
bin_01
N in A or B: 3236
N common: 402
Overlap: 12.42%
N only in A (B): 1301
--------------------------------------------------
bin_02
N in A or B: 3350
N common: 288
Overlap: 8.60%
N only in A (B): 1301
--------------------------------------------------
bin_03
N in A or B: 3120
N common: 518
Overlap: 16.60%
N only in A (B): 1301
--------------------------------------------------


# Figs distinct anomalies

In [50]:
plt.ioff()

for bin_id in bins_ids:

    print(bin_id)

    score_df = scores_df_dict[bin_id]

    (
        ids_lof, ids_iforest,
        common_ids,
        only_in_lof_ids, only_in_iforest_ids
    ) = AnomalyOverlapAnalyzer.overlap_pair_scores(
        score_a='lof_score',
        score_b='iforest_score',
        df=score_df.copy(),
        quantile=99
    )
    
    for score in ['lof_score', 'iforest_score']:

        if score == 'lof_score':
            specids = list(only_in_lof_ids)
            rank_col = 'rank_lof'
            save_to = f"{latent_dir}/{bin_id}/figs/lof_only"
        else:
            specids = list(only_in_iforest_ids)
            rank_col = 'rank_iforest'
            save_to = f"{latent_dir}/{bin_id}/figs/iforest_only"


        specids = np.array(specids, dtype=int)

        ranks = score_df.loc[
            specids, rank_col
        ].to_numpy().astype(int)

        specs = np.empty((len(specids), wave.size))

        for i, objid in enumerate(specids):

            spec_idx = specobjid_to_idx(
                objid, idx_id_spec
            )

            specs[i, :] = spectra[spec_idx, :]

        # -----------------------------------------------------------
        print(save_to)
        os.makedirs(save_to, exist_ok=True)

        anomaly_plot(
            wave_nm, specs=specs,
            objids=specids, ranks=ranks,
            save_to=save_to
        )

bin_00
/home/elom/phd/code/latent/bin_00/figs/lof_only
/home/elom/phd/code/latent/bin_00/figs/iforest_only
bin_01781
/home/elom/phd/code/latent/bin_01/figs/lof_only
/home/elom/phd/code/latent/bin_01/figs/iforest_only
bin_02113
/home/elom/phd/code/latent/bin_02/figs/lof_only
/home/elom/phd/code/latent/bin_02/figs/iforest_only
bin_03108
/home/elom/phd/code/latent/bin_03/figs/lof_only
/home/elom/phd/code/latent/bin_03/figs/iforest_only


# Fig Common anomalies

In [52]:
plt.ioff()

for bin_id in bins_ids:

    score_df = scores_df_dict[bin_id]

    (
        ids_lof, ids_iforest,
        common_ids,
        only_in_lof_ids, only_in_iforest_ids
    ) = AnomalyOverlapAnalyzer.overlap_pair_scores(
        score_a='lof_score',
        score_b='iforest_score',
        df=score_df.copy(),
        quantile=99
    )
    
    rank_lof_col = 'rank_lof'
    rank_iforest_col = 'rank_iforest'

    specids = list(common_ids)
    specids = np.array(specids, dtype=int)

    ranks_lof = score_df.loc[
        specids, rank_lof_col
    ].to_numpy().astype(int)

    ranks_iforest = score_df.loc[
        specids, rank_iforest_col
    ].to_numpy().astype(int)

    specs = np.empty((len(specids), wave.size))

    for i, objid in enumerate(specids):

        spec_idx = specobjid_to_idx(
            objid, idx_id_spec
        )

        specs[i, :] = spectra[spec_idx, :]

    # -----------------------------------------------------------
    save_to = f"{latent_dir}/{bin_id}/figs/common"
    print(save_to)
    os.makedirs(save_to, exist_ok=True)

    common_anomaly_plot(
        wave_nm, specs=specs,
        objids=specids,
        ranks_lof=ranks_lof, ranks_iforest=ranks_iforest,
        save_to=save_to
    )

/home/elom/phd/code/latent/bin_00/figs/common
/home/elom/phd/code/latent/bin_01/figs/common
/home/elom/phd/code/latent/bin_02/figs/common
/home/elom/phd/code/latent/bin_03/figs/common
